[Guide Index](README.md)


# 06.04 SRL (SPARQL Rules) as RDF

A parsed SRL rule set — `srl.RuleSet`, the same dataclass tree `srl.parse_ruleset()` returns (see the [SRL (SPARQL Rules) guide](04.01-srl-sparql-rules.ipynb) for the rules language itself) — can also be represented as an RDF graph, under the `srl:` namespace, the same "syntax-tree-as-RDF" pattern used elsewhere in StarLayer (e.g. [SPARQL queries as RDF](06.03-sparql-query-as-rdf.ipynb), [OWL Manchester Syntax as RDF](06.05-manchester-syntax-as-rdf.ipynb)). This lets a rule set be stored, inspected, or edited as ordinary RDF, and validated structurally with SHACL.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later sections reuse variables from earlier ones.


## 06.04.1 A parsed rule set

The same base graph and rule set as the [SRL (SPARQL Rules) guide](04.01-srl-sparql-rules.ipynb) - see that guide for what these rules mean; this one is only about representing the already-parsed result as RDF.

The graphs here are `StarLayerGraph`, the Oxigraph-backed graph class. A fresh `StarLayerGraph` has no prefixes bound, so the cells bind the ones they print with.


In [1]:
from starlayer.graph import Namespace, RDF
from starlayer.graph import StarLayerGraph
from starlayer.sparql import srl

EX = Namespace("http://example.org/")

# StarLayerGraph is the Oxigraph-backed graph class (see the Oxigraph-backed graph guide)
base = StarLayerGraph()
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")
base.bind("ex", EX)  # a fresh StarLayerGraph has no prefixes of its own; bind what you need

rules_text = """
PREFIX ex: <http://example.org/>

RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
print(f"{len(ruleset.rules)} rules parsed")


4 rules parsed


/Users/johnclements/Documents/GitHub/hidden-graph/starlayer/packages/starlayer/starlayer/shacl/_adapter.py:210: StarLayerFidelityWarning: The data graph does not preserve literal lexical forms (Oxigraph normalises them on storage: "04"^^xsd:integer becomes "4", "1"^^xsd:boolean becomes "true"), and the shapes use constraints whose result depends on the original lexical form or datatype: sh:datatype. Results may differ from validating the same data with its lexical forms as written. See StarLayer issue #190; upstream oxigraph#526, oxigraph#1989, PR #2010.
  outcome = api.validate(


## 06.04.2 Encode the rule set as RDF

`ruleset_to_tree()` encodes a `RuleSet` as RDF under the `srl:` namespace - the same tree-RDF pattern `sqe:` uses for SPARQL queries (see the [SPARQL queries as RDF guide](06.03-sparql-query-as-rdf.ipynb)). It encodes into the graph you pass (here an empty `StarLayerGraph`, so the tree lives in Oxigraph) and returns `(graph, root)` - `root` is the `srl:RuleSet` node's own identity, needed later to decode a specific rule set back out of the graph.

Each rule, and each `NOT`, carries the spec's flag (`evaldata` until the draft of 2026-11-19, `evalground` now; the `srl:` names keep the older word) as `srl:ruleEvaldata` or `srl:negationEvaldata`: `true` means the patterns match the evaluation graph (a plain `WHERE` or `NOT`), `false` means they match only the base graph (`WHERE GROUND` or `NOT GROUND`; see the [SRL (SPARQL Rules) guide](04.01-srl-sparql-rules.ipynb)).

In [2]:
from starlayer.sparql.vocab import SPEXPR

srl_graph, root = srl.ruleset_to_tree(ruleset, StarLayerGraph())

srl_graph.bind("srl", srl.SRL)
srl_graph.bind("spexpr", SPEXPR)
srl_graph.bind("ex", EX)

print("ruleset as graph:")
print(srl_graph.serialize(format="turtle", pretty=True))

ruleset as graph:
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix srl: <https://github.com/hidden-graph/starlayer/ns/sparql/srl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

_:b0 a srl:RuleSet ;
    srl:data () ;
    srl:imports () ;
    srl:rules (
        [
            a srl:Rule ;
            srl:body (
                [
                    a srl:TriplePattern ;
                    srl:object "v"^^spexpr:Variable ;
                    srl:predicate ex:hasVulnerability ;
                    srl:subject "x"^^spexpr:Variable
                ]
            ) ;
            srl:head (
                [
                    a srl:TriplePattern ;
                    srl:object "v"^^spexpr:Variable ;
                    srl:predicate ex:exposedTo ;
                    srl:subject "x"^^spexpr:Variable
                ]
            ) ;
            srl:r

## 06.04.3 Edit the rule set by editing the RDF

The rule set graph is an ordinary `StarLayerGraph` and can be edited as one (`set(triple)` replaces the value a subject has for a predicate; `remove` followed by `add` does the same for one value). Below, rule 2's own body - `?x ex:dependsOn ?y . ?y ex:exposedTo ?v` - is edited so the dependency edge it walks is `ex:reliesOn` instead of `ex:dependsOn`, directly as RDF graph surgery, no text-level find/replace anywhere.

In [3]:
# a triple pattern's own predicate is located and replaced
for s in list(srl_graph.subjects(srl.SRL.predicate, EX.dependsOn)):
    srl_graph.remove((s, srl.SRL.predicate, None))
    srl_graph.add((s, srl.SRL.predicate, EX.reliesOn))

print("edited: rule 2's body now walks ex:reliesOn, not ex:dependsOn")

edited: rule 2's body now walks ex:reliesOn, not ex:dependsOn


## 06.04.4 Decode back into a real `RuleSet`, and run it

`tree_to_ruleset()` turns the edited RDF back into a real `RuleSet`. The base data's dependency chain is `frontend -> app -> db`, with the only vulnerability on `db` itself - running both rule sets against the *same* base graph proves the edit is semantic: breaking the `ex:dependsOn` propagation edge stops `app`/`frontend` from inheriting `db`'s exposure, so only `db` stays `criticallyExposed`.

In [4]:
edited_ruleset = srl.tree_to_ruleset(srl_graph, root)

original_inferred = ruleset.infer(base)
edited_inferred = edited_ruleset.infer(base)

print("original criticallyExposed:", sorted(base.qname(s) for s in original_inferred.subjects(EX.status, EX.criticallyExposed)))
print("edited criticallyExposed:  ", sorted(base.qname(s) for s in edited_inferred.subjects(EX.status, EX.criticallyExposed)))

original criticallyExposed: ['ex:app', 'ex:db', 'ex:frontend']
edited criticallyExposed:   ['ex:db']


## 06.04.5 Export the modified rule set back to text

`ruleset_to_text()` renders the edited `RuleSet` back to SRL text. Unlike `sqe_tree_to_text()`'s own `PREFIX` preservation, the parsed SRL syntax tree only ever holds fully-resolved absolute IRIs - no original `PREFIX`/`BASE` declaration survives to render back. Passing `namespace_manager=` abbreviates the output; it accepts a `StarLayerGraph` (here the base graph, so its `ex:` binding is used), a prefix-to-namespace `dict` or an rdflib `NamespaceManager`. Only prefixes that were bound are used, which is why `rdf:type` stays a full IRI in the output below (the base graph has no `rdf:` binding); without the argument every IRI renders in full.

In [5]:
print(srl.ruleset_to_text(edited_ruleset, namespace_manager=base))

RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:reliesOn ?y . ?y ex:exposedTo ?v . }
RULE { ?x ex:status ex:criticallyExposed . } WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= "9"^^<http://www.w3.org/2001/XMLSchema#decimal>) }
RULE { ?x ex:status ex:safeToDeploy . } WHERE { ?x <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> ex:Component . NOT { ?x ex:status ex:criticallyExposed . } }


## 06.04.6 Validate

`parse_ruleset()` returns a rule set for any text the SRL grammar accepts, so a rule set with problems can still be loaded, inspected and fixed. It raises only for text that is not SRL. `srl_validate()` reports the problems in the rule set's syntax tree. It reports the §4.2 well-formedness conditions (an unbound head, filter or assignment variable, a `SET(...)` reusing an already-bound variable, an aggregate used inside `FILTER` or `SET`), the structure checked by the `srl:` SHACL shapes, and a rule set that cannot be put in an evaluation order. Each §4.2 problem is a result on the element at fault, with a message that names the variable. A rule set that has `IMPORTS` that are not resolved gets a warning, which does not make the report fail.

`srl_validate()` is also needed whenever a tree is edited by hand, since those edits (like section 06.04.3's above) happen outside of `parse_ruleset()`. A rule set that breaks §4.2 is also refused when it is run: `infer()` raises a `FormatError` (kind `"srl"`).

Below, `srl_validate()` checks the edited syntax tree from section 06.04.3, and it conforms.

In [6]:
# validate the edited syntax tree from section 06.04.3 above
report = srl.srl_validate(srl_graph)
print("conforms:", report.conforms)

conforms: True


Editing the tree can also make it invalid. Removing a `srl:TriplePattern`'s own required `srl:predicate` is one example - `srl_validate()` catches it:

In [7]:
# break the tree: remove a triple pattern's own required predicate
pattern_node = next(srl_graph.subjects(RDF.type, srl.SRL.TriplePattern))
srl_graph.remove((pattern_node, srl.SRL.predicate, None))

broken_report = srl.srl_validate(srl_graph)
print("conforms:", broken_report.conforms)
print(broken_report.report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:maxCount 1 ; sh:message "srl:TriplePattern must have exactly one srl:predicate, an IRI/Variable" ; sh:minCount 1 ; sh:node srl:PredicateOrVariableShape ; sh:path srl:predicate ]
	Focus Node: [ <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> srl:BodyElement, srl:GroupBodyElement, srl:NegationBodyElement, srl:TriplePattern ; srl:object ex:criticallyExposed ; srl:subject "x"^^spexpr:Variable ]
	Result Path: srl:predicate
	Message: srl:TriplePattern must have exactly one srl:predicate, an IRI/Variable



### 06.04.6.1 A rule set that's well-formed SPARQL-wise, but fails SRL's own §4.2 check

`RULE { ?x ex:status ?z } WHERE { ?x a ex:Component . }` parses as a perfectly ordinary rule - but `?z` in the head is never bound anywhere in the body. §4.2's well-formedness condition requires every head variable to appear in the body's own bound-variable set; `?z` doesn't, so this signals a typo or an incomplete rule, the same "output with no matching input" shape as `sqe:`'s own unbound-projected-variable check in the [SPARQL queries as RDF guide](06.03-sparql-query-as-rdf.ipynb).

Below, the text is parsed into a rule set, which loads as usual. `srl_validate()` then reports the unbound head variable, and the message of each result is printed. Running the rule set raises a `FormatError`, so the typo does not give a wrong answer later.

In [8]:
from starlayer.errors import FormatError

bad_rules = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:status ?z } WHERE { ?x a ex:Component . }
"""

# the rule set loads; srl_validate() reports what is wrong with its tree
bad_ruleset = srl.parse_ruleset(bad_rules)
bad_tree, bad_root = srl.ruleset_to_tree(bad_ruleset)
bad_report = srl.srl_validate(bad_tree)
print("conforms:", bad_report.conforms)
# the report holds one result for each problem; each message names the variable
for row in bad_report.report_graph.query("SELECT ?message WHERE { ?result <http://www.w3.org/ns/shacl#resultMessage> ?message }"):
    print(row.message.value)

# running it is refused
try:
    bad_ruleset.infer(StarLayerGraph())
except FormatError as exc:
    print("infer() raised:", exc)

conforms: False
The rule head uses the variable z, which the rule body never binds.
infer() raised: SRL: rule set is not well-formed per §4.2: unbound_head_variable (?z in rule head)
